In [43]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-m3', device='cuda:4')

In [44]:
query_text = "query: Where is Korea University?"
positive_text = "document: Korea University is located in South Korea, Seoul"

model.eval()
query_embedding = model.encode(query_text, convert_to_tensor=True)
positive_embedding = model.encode(positive_text, convert_to_tensor=True)

In [45]:
import torch
import torch.nn.functional as F

embedding_dim = 1024
torch.manual_seed(42)

P = torch.nn.Parameter(torch.randn(embedding_dim)).to('cuda:4')

print(f"Query Embedding Shape: {query_embedding.shape}")
print(f"Positive Embedding Shape: {positive_embedding.shape}")
print(f"Parameter P Shape: {P.shape}")

Query Embedding Shape: torch.Size([1024])
Positive Embedding Shape: torch.Size([1024])
Parameter P Shape: torch.Size([1024])


In [46]:
import torch
with torch.no_grad():
    target_interaction = query_embedding * positive_embedding
    predicted_interaction = query_embedding * P

In [47]:
log_pred_dist = F.log_softmax(predicted_interaction, dim=-1)
target_dist = F.softmax(target_interaction, dim=-1)

kl_loss = F.kl_div(log_pred_dist, target_dist, reduction='batchmean')

print(f"Target Interaction Shape: {target_interaction.shape}")
print(f"Predicted Interaction Shape: {predicted_interaction.shape}")
print("-" * 30)
print(f"계산된 KL Divergence Loss: {kl_loss.item()}")

Target Interaction Shape: torch.Size([1024])
Predicted Interaction Shape: torch.Size([1024])
------------------------------
계산된 KL Divergence Loss: 4.179773895884864e-07


In [48]:
temperature = 1e-5

log_pred_dist = F.log_softmax(predicted_interaction / temperature, dim=-1)
target_dist = F.softmax(target_interaction / temperature, dim=-1)

kl_loss = F.kl_div(log_pred_dist, target_dist, reduction='batchmean')

print(f"Target Interaction Shape: {target_interaction.shape}")
print(f"Predicted Interaction Shape: {predicted_interaction.shape}")
print("-" * 30)
print(f"계산된 KL Divergence Loss: {kl_loss.item()}")



Target Interaction Shape: torch.Size([1024])
Predicted Interaction Shape: torch.Size([1024])
------------------------------
계산된 KL Divergence Loss: 17.092227935791016


In [ ]:
import torch.nn as nn
loss_fn = nn.MSELoss()
loss = loss_fn(predicted_interaction * 100, target_interaction * 100)

print(loss)

tensor(8.6071, device='cuda:4')


: 

In [51]:
loss_fn = nn.L1Loss()
loss = loss_fn(predicted_interaction, target_interaction)

print(loss)

tensor(0.0188, device='cuda:4')
